<a href="https://colab.research.google.com/github/Anushka-1431/PRODIGY_GA_01/blob/main/PRODIGY_GA_01_GPT2_Text_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [ ]:
!pip install transformers datasets torch accelerate

In [ ]:
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

from datasets import Dataset

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [ ]:
ai_paragraphs = []

topics = [
    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Neural Networks",
    "Natural Language Processing",
    "Computer Vision",
    "Robotics",
    "Data Science",
    "Generative AI",
    "AI in Healthcare",
    "AI in Education",
    "AI in Finance",
    "AI Ethics",
    "Autonomous Vehicles",
    "Predictive Analytics"
]

for i in range(1000):
    topic = topics[i % len(topics)]

    paragraph = f"""
{topic} is transforming modern industries through intelligent automation and data-driven decision making.
Organizations use {topic.lower()} to improve efficiency, reduce costs, and deliver better customer experiences.
Researchers continue to develop advanced algorithms that increase the accuracy and reliability of AI systems.
The adoption of {topic.lower()} is growing rapidly across healthcare, education, finance, manufacturing, and transportation.
Future innovations in {topic.lower()} are expected to create new opportunities while addressing complex global challenges.
"""
    ai_paragraphs.append(paragraph)

with open("dataset.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(ai_paragraphs))

print("Dataset Created Successfully")
print("Total paragraphs:", len(ai_paragraphs))

Dataset Created Successfully
Total paragraphs: 1000


In [ ]:
with open("dataset.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Characters:", len(text))
print(text[:500])

Characters: 588407

Artificial Intelligence is transforming modern industries through intelligent automation and data-driven decision making.
Organizations use artificial intelligence to improve efficiency, reduce costs, and deliver better customer experiences.
Researchers continue to develop advanced algorithms that increase the accuracy and reliability of AI systems.
The adoption of artificial intelligence is growing rapidly across healthcare, education, finance, manufacturing, and transportation.
Future innovat


In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer Loaded Successfully


In [ ]:
from transformers import GPT2LMHeadModel

print("GPT2LMHeadModel Imported")

GPT2LMHeadModel Imported


In [ ]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

print("GPT-2 Model Loaded Successfully")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 Model Loaded Successfully


In [ ]:
!ls

dataset.txt  drive  sample_data


In [ ]:
with open("dataset.txt", "r") as f:
    text = f.read()

dataset = Dataset.from_dict({
    "text": [text]
})

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 1
})


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function)

print("Tokenization Successful")

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenization Successful


In [ ]:
print(tokenized_dataset[0])

{'text': '\nArtificial Intelligence is transforming modern industries through intelligent automation and data-driven decision making.\nOrganizations use artificial intelligence to improve efficiency, reduce costs, and deliver better customer experiences.\nResearchers continue to develop advanced algorithms that increase the accuracy and reliability of AI systems.\nThe adoption of artificial intelligence is growing rapidly across healthcare, education, finance, manufacturing, and transportation.\nFuture innovations in artificial intelligence are expected to create new opportunities while addressing complex global challenges.\n\n\nMachine Learning is transforming modern industries through intelligent automation and data-driven decision making.\nOrganizations use machine learning to improve efficiency, reduce costs, and deliver better customer experiences.\nResearchers continue to develop advanced algorithms that increase the accuracy and reliability of AI systems.\nThe adoption of machin

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

tokenized_dataset = tokenized_dataset.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1
})


In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data Collator Created Successfully")

Data Collator Created Successfully


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    save_steps=10,
    save_total_limit=2,
    logging_steps=1,
    report_to="none"
)

print("Training Arguments Created Successfully")

Training Arguments Created Successfully


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer Created Successfully")

Trainer Created Successfully


In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,2.537041
2,2.316143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2, training_loss=2.4265918731689453, metrics={'train_runtime': 13.3931, 'train_samples_per_second': 0.149, 'train_steps_per_second': 0.149, 'total_flos': 130646016000.0, 'train_loss': 2.4265918731689453, 'epoch': 2.0})

In [ ]:
model.save_pretrained("fine_tuned_gpt2")
tokenizer.save_pretrained("fine_tuned_gpt2")

print("Model Saved Successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully


In [ ]:
!ls fine_tuned_gpt2

config.json		model.safetensors      tokenizer.json
generation_config.json	tokenizer_config.json


In [ ]:
from transformers import pipeline

# Create text generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Generate 3 outputs
for i in range(3):
    result = generator(
        "Artificial Intelligence",
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8
    )

    print(f"\n{'='*50}")
    print(f"Generated Output {i+1}")
    print(f"{'='*50}\n")

    print(result[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Output 1

Artificial Intelligence and the "The Best of Both Worlds" exhibit.

The "The Best of Both Worlds" exhibit is known as the "The Best of Both Worlds" exhibit, which contains an examination of AI and the impact of AI technologies on reality.


Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Output 2

Artificial Intelligence Intelligence (AI) at http://www.sciencedaily.com/releases/2016/07/delta-8-6.html.

The Pentagon also stated:

"The FBI is conducting an extensive review of the

Generated Output 3

Artificial Intelligence and Artificial Intelligence will provide critical information, enabling us to make the right decisions in a competitively.

The goal for Artificial Intelligence and Artificial Intelligence will be to provide a high level of accuracy in the way the human mind is being used by us


In [ ]:
print(len(text))

588407
